# ⚖️ Thử nghiệm Tuning Alpha cho Hybrid Search
**Mục tiêu:** Tìm ra giá trị `alpha` tối ưu nhất cho `HybridRetriever`.
- `alpha = 1.0` → Chỉ dùng Qdrant Semantic Search (Dense)
- `alpha = 0.0` → Chỉ dùng BM25 Keyword Search (Sparse)
- `alpha = 0.6` → Kết hợp 60% Semantic + 40% BM25 (giá trị mặc định)

Ta sẽ chạy với bộ câu hỏi mẫu và đo Precision@K để chọn alpha tốt nhất.

In [ ]:
import os, sys, json
from dotenv import load_dotenv

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

from rank_bm25 import BM25Okapi
from source.core.config import Settings
from source.retrieval.hybrid_retriever import HybridRetriever

settings = Settings()
retriever = HybridRetriever(settings=settings, collection_name="Traffic_Law_Hybrid")
print("✅ Đã khởi tạo HybridRetriever!")

## 1. Load BM25 data từ chunks

In [ ]:
CHUNKS_PATH = os.path.join(PROJECT_ROOT, 'Data', 'chunks', 'traffic_chunks.json')
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    retriever.corpus_chunks = json.load(f)

tokenized_corpus = [retriever._tokenize(c['content']) for c in retriever.corpus_chunks]
retriever.bm25 = BM25Okapi(tokenized_corpus)

print(f"✅ Đã tải {len(retriever.corpus_chunks):,} chunks vào BM25")
print("⚠️  Lưu ý: Qdrant Dense Search cần có collection 'Traffic_Law_Hybrid' đang chạy.")
print("   Nếu chưa index → alpha=1.0 sẽ lỗi, nhưng alpha=0.0 (BM25 only) vẫn chạy được.")

## 2. Tạo bộ test câu hỏi + điều luật mong đợi

In [ ]:
# Ground truth: câu hỏi → điều luật mong đợi trong kết quả truy xuất
TEST_CASES = [
    {
        "query": "xe máy uống rượu bị phạt bao nhiêu?",
        "expected_keywords": ["nồng độ cồn", "điều 6", "100/2019"],
        "expected_dieu": "Điều 6"
    },
    {
        "query": "vượt đèn đỏ ô tô phạt bao nhiêu?",
        "expected_keywords": ["tín hiệu", "đèn", "điều 5"],
        "expected_dieu": "Điều 5"
    },
    {
        "query": "đi xe máy không đội mũ bảo hiểm phạt tiền bao nhiêu?",
        "expected_keywords": ["mũ bảo hiểm", "điều 11"],
        "expected_dieu": "Điều 11"
    },
    {
        "query": "tốc độ tối đa cho phép đường cao tốc là bao nhiêu?",
        "expected_keywords": ["tốc độ", "km/h", "cao tốc"],
        "expected_dieu": "Điều 7"
    },
    {
        "query": "xe không có giấy phép lái xe bị phạt gì?",
        "expected_keywords": ["giấy phép lái xe", "gplx"],
        "expected_dieu": "Điều 21"
    },
]
print(f"✅ Đã tạo {len(TEST_CASES)} test cases")

## 3. Hàm đánh giá và chạy benchmark theo alpha

In [ ]:
def keyword_precision(results, expected_keywords, k=5):
    """Đo tỷ lệ chunk top-K chứa ít nhất 1 từ khoá mong đợi."""
    top_k = results[:k]
    hits = 0
    for r in top_k:
        content_lower = r['chunk']['content'].lower()
        meta_dieu = r['chunk']['metadata'].get('dieu', '').lower()
        for kw in expected_keywords:
            if kw.lower() in content_lower or kw.lower() in meta_dieu:
                hits += 1
                break
    return hits / k

def run_benchmark_bm25_only(test_cases, top_k=5):
    """Chạy BM25 thuần (không cần Qdrant)."""
    import numpy as np
    scores_list = []
    for tc in test_cases:
        tokenized_q = retriever._tokenize(tc['query'])
        bm25_scores = retriever.bm25.get_scores(tokenized_q)
        if np.max(bm25_scores) > 0:
            bm25_scores = bm25_scores / np.max(bm25_scores)
        top_indices = bm25_scores.argsort()[-top_k:][::-1]
        results = [{"chunk": retriever.corpus_chunks[i], "score": float(bm25_scores[i])} for i in top_indices]
        p = keyword_precision(results, tc['expected_keywords'], top_k)
        scores_list.append(p)
    return sum(scores_list) / len(scores_list)

def run_benchmark_hybrid(test_cases, alpha, top_k=5):
    """Chạy Hybrid Search với alpha cho trước."""
    scores_list = []
    for tc in test_cases:
        try:
            results = retriever.search(tc['query'], top_k=top_k, alpha=alpha)
            p = keyword_precision(results, tc['expected_keywords'], top_k)
            scores_list.append(p)
        except Exception as e:
            print(f"  ⚠️  Lỗi ở alpha={alpha}: {e} (bỏ qua câu hỏi này)")
            scores_list.append(0.0)
    return sum(scores_list) / len(scores_list) if scores_list else 0.0

print("✅ Các hàm đánh giá đã sẵn sàng!")

## 4. Chạy benchmark BM25 Only (không cần Qdrant)

In [ ]:
print("🔍 Đang chạy BM25 Only benchmark...")
bm25_score = run_benchmark_bm25_only(TEST_CASES)
print(f"\n📊 KẾT QUẢ BM25 Only (alpha=0.0):")
print(f"   Keyword Precision@5: {bm25_score:.3f} ({bm25_score*100:.1f}%)")

# Chi tiết từng câu hỏi
import numpy as np
print("\n📋 Chi tiết từng câu hỏi:")
for tc in TEST_CASES:
    tokenized_q = retriever._tokenize(tc['query'])
    bm25_scores = retriever.bm25.get_scores(tokenized_q)
    top_idx = bm25_scores.argsort()[-3:][::-1]
    print(f"  Q: {tc['query']}")
    for idx in top_idx:
        c = retriever.corpus_chunks[idx]
        print(f"    [{bm25_scores[idx]:.3f}] {c['metadata']['dieu']} | {c['content'][:80]}...")
    print()

## 5. Benchmark Hybrid với các giá trị alpha khác nhau (cần Qdrant)

In [ ]:
ALPHA_VALUES = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]

print("🚀 Đang benchmark Hybrid Search với các alpha values...")
print("   (Cần Qdrant đang chạy và đã có collection 'Traffic_Law_Hybrid')\n")

alpha_results = {}
for alpha in ALPHA_VALUES:
    score = run_benchmark_hybrid(TEST_CASES, alpha=alpha)
    alpha_results[alpha] = score
    bar = '█' * int(score * 20)
    print(f"  alpha={alpha:.1f}: {score:.3f} ({score*100:.1f}%)  {bar}")

best_alpha = max(alpha_results, key=alpha_results.get)
print(f"\n🏆 Alpha tối ưu: {best_alpha} (Precision@5 = {alpha_results[best_alpha]:.3f})")
print(f"💡 Cập nhật giá trị này vào simple_chat.py: retriever.search(query, alpha={best_alpha})")